# Checkpoint B solutions: a full agent on a track and a car it has never met

These reference solutions close **Part 2**.
They follow the brief's discipline: measure the race-day track and plan it on paper (Questions 1–2), read the reference's family record with both of Unit 11's rules before racing anything (Question 3), tune on the track × car family with the small protocol and freeze (Question 4), and only then check the frozen file and race the five sealed cars, once (Question 5); Question 6 reads the race-day record.
The submission is the book's stand-in, `agents/unit-11.py`, as the statements say.
Every race is seeded and timing-off, so every number below is reproducible, and executable asserts check each answer: the planning answers against an independent derivation, the race answers against the book's pinned values (to one control tick, 0.05 s).
Every number is a number about racekit's **carla-like car**: a model, not the truth.

**The brief.**
Race day.
A track you have never driven, `test-circuit-seed2027`, and a car you have never met: five cars of the perturbed family, sealed (`perturb=1000` … `1004`).
You may measure the track with the carla-like car you know, solve its line, and tune on the five training tracks with the training cars.
You may race the race-day track once: the submission check, then five scored races, after every choice is frozen.

Unit 11 left two findings open.
A margin bought on one track did not carry to another (`test-circuit-seed2026`), and the fraction it shipped was the one with the shortest **expected race time** on its training family.
Today both are put to the test, on a track and on cars that are in no family you have raced.

The tracks and the cars, all built by racekit:

- **the training tracks**, five tracks you may race as often as you like, each with its own line and its own measured grip: `test-circuit`, `test-circuit-seed1`, `test-circuit-seed2`, `test-circuit-seed3` and `test-circuit-seed4`;
- **the training cars**, cars 0–23 of the perturbed family (`carla_like(noise=True, perturb=k)`), whose factors you may read with `perturbation_factors(k)`;
- **the race-day track**, `test-circuit-seed2027`: measure it (Question 1), solve its line and plan it (Question 2), and race it only on race day (Question 5);
- **the race-day cars**, the sealed cars `perturb=1000` … `1004`, raced once each, on race day.

## The rules

- **Your submission is your own Unit 11 submission.**
  The setup cell points `SUBMISSION` at `agents/unit-11.py`, the book's Unit 11 reference, as a stand-in: replace that path with your own Unit 11 file before you begin.
- **Three constants change, and nothing else:** `MU_G` (the race-day track's measured grip, Question 1), `LINE_CONTROL` (its line, Question 2) and `MU_FRACTION` (tuned on the family, Question 4).
  Every other constant stays as your Unit 11 file ships it.
- **The seal is an honour rule.** The sealed cars are built in one place only, the race-day cell of Question 5, inside a race.
  Never inspect them by any route: no `perturbation_factors` on them, no attribute of a car built from them, no re-deriving their factors from racekit's published algorithm, and no race with them before race day.
  Tune on the training cars 0–23 only.
- **No race on `test-circuit-seed2027`** except the six grip probes (Question 1) and race day (Question 5).
  Loading it, solving its line and planning on it are allowed.
  Race day starts with `check_submission` on your frozen file, which races racekit's default kinematic car on the race-day track with the timing check; it must pass before any sealed car races there.
- **Timing.** Every race of your submission in this notebook uses a **timing-off copy** of it, `MPC_BUDGET_MS = None` written in with `write_variant` (the setup's `NO_TIMING`, as in Unit 11), so the numbers reproduce on any machine.
  The frozen file itself keeps `MPC_BUDGET_MS = 5.0`, and it is the file `check_submission` validates: that check is the one run with the timing on.
- **The clean rule** (Unit 9's, from the race's own line): a race is clean when it finished, with 0 collisions, and its max |cross-track error from the line it raced| over the driving rows is at most 2 m.
- **Every three-lap race** uses `carla_like(noise=True, perturb=k)` and `max_seconds=700.0`; the race seed is 0 for every tuning race of Question 4 and k mod 5 for sealed car k.

**The bar:** all five race-day races clean, and their median race time at most **300 s**.
The race is pass/fail on the bar.
Your evidence and your written answers carry the rest of the grade: a submission that misses the bar after an honest Question 4 can still earn full marks, if the write-up says what the record showed.

Run the setup cell first.
Each question names the variables your answer must define; fill in every `None`, and write each written answer in the markdown cell under its prompt.
Run the questions in order: later questions reuse earlier functions and results.
A race takes a few seconds (a perturbed MPC race about 6 s); no cell should run for more than about two minutes, so the long sweeps are split one fraction per cell.

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like, perturbation_factors
from racekit.check_submission import check_submission
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import grip_sections, load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
SUBMISSION = AGENTS / "unit-11.py"  # a stand-in: replace it with the path of YOUR Unit 11 submission
PROBE_AGENT = AGENTS / "unit-07.py"  # the book's Unit 7 reference: Unit 8's grip-probe driver (Question 1 only)
workdir = Path(mkdtemp(prefix="roar-checkpoint-b-sol-"))

TRAINING = ("test-circuit", "test-circuit-seed1", "test-circuit-seed2", "test-circuit-seed3", "test-circuit-seed4")
TRAINING_CARS = tuple(range(24))  # the perturbed family's training cars: the only cars you tune on
RACE_DAY_TRACK = "test-circuit-seed2027"  # raced only by the six probes (Question 1) and on race day (Question 5)
NO_TIMING = {"MPC_BUDGET_MS": None}  # every race here: no wall clock inside a race (Unit 11)
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
MAX_SECONDS = 700.0  # s: every three-lap race's time cap
TARGET_S = 300.0  # s: race day's target median
PROTOCOL_SEED = 0  # Question 4: the race seed of every tuning race
START_FRACTION = 0.54  # Unit 11's shipped MU_FRACTION: Question 4 starts here
FRACTION_STEP = 0.02  # the fraction grid (Units 9-11)
TICK = 0.05 + 1e-9  # s: one control tick (race times are pinned to one tick)
CEILING, FLOOR = 0.60, 0.40  # Question 4 never races a fraction above CEILING or below FLOOR
A_DEC, A_ACC, TOP_SPEED = 5.0, 4.0, 45.0  # m/s^2, m/s^2, m/s: the reference's profile (use yours if they differ)
MARGIN, PIN = 2.5, 30  # m kept from each edge, and waypoints at each end of the lap held on the centreline (Unit 9)

# Unit 8's grip probe (Question 1): PROBE_AGENT with A_LAT = PROBE_A_LAT and GRIP 1.0 except PROBE_KNOB in the
# probed section; one lap (laps=1) on the unperturbed carla_like(noise=True), race seed PROBE_SEED, PROBE_SECONDS cap.
PROBE_A_LAT, PROBE_KNOB, PROBE_SEED, PROBE_SECONDS = 3.5, 3.5, 0, 300.0
MIN_ROWS = 20  # a section with this many qualifying rows or fewer gives no estimate

# Unit 8's grip probes: the measured mu * g of each grip section (m/s^2, rounded to 0.01), per training track.
MEASURED_MU_G = {
    "test-circuit": (7.89, 9.72, 8.65, 7.87, 8.30, 8.14),
    "test-circuit-seed1": (7.72, 9.80, 9.36, 8.38, 9.53, 7.56),
    "test-circuit-seed2": (9.06, 8.37, 8.49, 8.86, 9.78, 9.92),
    "test-circuit-seed3": (9.55, 8.89, 8.15, 9.94, 8.50, 9.09),
    "test-circuit-seed4": (8.79, 8.53, 9.10, 8.54, 8.41, 9.97),
}

race_log = []  # every race you run with race(), as (track, car): your evidence (car None: the unperturbed car)


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, written into workdir.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def race(source, track, k=None, seed=0, laps=3, max_seconds=MAX_SECONDS):
    # Every race of this checkpoint: the carla-like car with its noise, car k of the perturbed family
    # (k None: the unperturbed car), one race seed.
    race_log.append((track, k))
    return run_race(source, track, car=carla_like(noise=True, perturb=k), seed=seed, laps=laps,
                    max_seconds=max_seconds)


def submission_module(source):
    # A submission's module as it ships (Unit 9's idiom): its constants are attributes, e.g. .LINE_CONTROL.
    with shim_installed():
        _, module_name = load_submission(source)
    return sys.modules[module_name]


# Units 2 and 9's tools, as Units 9-11 gave them.
def menger(x, y):
    # The signed Menger curvature at every point of a closed polyline (NumPy arrays; left turns positive).
    xa, ya, xc, yc = np.roll(x, 1), np.roll(y, 1), np.roll(x, -1), np.roll(y, -1)
    twice_area = (x - xa) * (yc - ya) - (y - ya) * (xc - xa)
    sides = np.hypot(x - xa, y - ya) * np.hypot(xc - x, yc - y) * np.hypot(xc - xa, yc - ya)
    return 2.0 * twice_area / sides


def spline_basis(n, step=10):
    # The n x (n // step) cubic B-spline basis: a line's offsets are spline_basis(n) @ its control values.
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    # The left unit normal at every point of a closed polyline, as two NumPy arrays (nx, ny).
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    # v[i+1] - 2 v[i] + v[i-1] along the first axis, the ends wrapping round.
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def line_offsets(control, n):
    # A line's offset (m, left positive) at each of the n waypoints, from its control values.
    return spline_basis(n) @ np.array(control, dtype=float)


# Unit 9's clean rule, from the race's own line (offsets: that line's per-waypoint offsets).
def line_error(result, offsets):
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result, offsets):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


# Your submission's test-circuit line and grip, as it ships them (test-circuit is raced as shipped).
shipped = submission_module(SUBMISSION)
SHIPPED_LINE = list(shipped.LINE_CONTROL)
SHIPPED_MU_G = list(shipped.MU_G)

## Question 1 - Measure the track

The race-day track's grip is not the grip of any track you have raced, so measure it the way Unit 8 did.

First write Unit 8's probe rule as a function.
`section_p99(telemetry, track, min_rows=MIN_ROWS)` returns a list of six pairs `(rows, p99)`, one per grip section, where:

- |a| at row i (for i from 1 to N − 2) is the size of the second difference of (x, y) at rows i − 1, i, i + 1, divided by 0.05²;
- row i qualifies when rows i − 1, i and i + 1 all have `held == 0`, both steps between them are shorter than 5 m, and `v` at row i is above 5 m/s;
- row i's section is `grip_sections(track)[(progress[i] + 1) % n]` (racekit's progress p is waypoint p + 1);
- `rows` is the number of qualifying rows in the section, and `p99` the 99th percentile of their |a| (`np.percentile(values, 99)`), or `None` when `rows <= min_rows`.

Then run Unit 8's six grip probes on `RACE_DAY_TRACK`, one per section.
The probe for section s is a variant of `PROBE_AGENT` with `A_LAT = PROBE_A_LAT` and `GRIP` 1.0 in every section except `PROBE_KNOB` in section s, raced with `race(source, RACE_DAY_TRACK, seed=PROBE_SEED, laps=1, max_seconds=PROBE_SECONDS)` (the unperturbed car).
From each probe keep only the probed section's pair.

Unit 8's rules for reading a probe:

- a probe that crashed (at least one collision) slid in its section, so its p99 is an **estimate** of μg there;
- a probe that never crashed gives a **lower bound**: the car did not need more grip than that, so the grip is at least that much;
- a section with `MIN_ROWS` qualifying rows or fewer gives **no estimate** (that would need another probe: ask your teacher before you go on).

The planner uses every value as measured, a lower bound included, as Unit 8 did with `monza-like`'s section 0.

Define:

- `section_p99`: the function;
- `race_day_track`: `load_track(RACE_DAY_TRACK)`;
- `probes`: a dict from each section 0 … 5 to its probe's `RaceResult`;
- `probe_outcome`: a dict from each section to `(finished, collisions)` of its probe;
- `probe_rows`: a dict from each section to the number of qualifying rows in the probed section;
- `race_day_mu_g`: a tuple of the six probed sections' p99, in section order, each rounded to 0.01 (m/s²): the race-day track's `MU_G`;
- `kind`: a dict from each section to `"estimate"`, `"lower bound"` or `"no estimate"`, by Unit 8's rules;
- `probe_races`: the number of entries in `race_log` on `RACE_DAY_TRACK` (it should be exactly 6).

In [ ]:
def section_p99(telemetry, track, min_rows=MIN_ROWS):
    # Unit 8's probe rule: |a| from the second difference of (x, y) at 0.05 s, on qualifying rows, per section.
    x, y, held, v, progress = (telemetry["x"], telemetry["y"], telemetry["held"], telemetry["v"],
                               telemetry["progress"])
    sections = grip_sections(track)
    n = len(track)
    values = [[] for _ in range(6)]
    for i in range(1, len(x) - 1):
        driving = held[i - 1] == 0 and held[i] == 0 and held[i + 1] == 0
        short_steps = (math.hypot(x[i] - x[i - 1], y[i] - y[i - 1]) < 5.0
                       and math.hypot(x[i + 1] - x[i], y[i + 1] - y[i]) < 5.0)
        if driving and short_steps and v[i] > 5.0:
            a = math.hypot(x[i + 1] - 2.0 * x[i] + x[i - 1], y[i + 1] - 2.0 * y[i] + y[i - 1]) / 0.05 ** 2
            values[sections[(progress[i] + 1) % n]].append(a)
    return [(len(rows), float(np.percentile(rows, 99)) if len(rows) > min_rows else None) for rows in values]


def probe_source(section):
    # Unit 8's probe for one section: PROBE_AGENT at A_LAT 3.5, GRIP 1.0 except PROBE_KNOB in that section.
    grip = [1.0] * 6
    grip[section] = PROBE_KNOB
    return variant(PROBE_AGENT, f"probe-{section}", A_LAT=PROBE_A_LAT, GRIP=tuple(grip))


race_day_track = load_track(RACE_DAY_TRACK)
probes = {section: race(probe_source(section), RACE_DAY_TRACK, seed=PROBE_SEED, laps=1, max_seconds=PROBE_SECONDS)
          for section in range(6)}
probe_outcome = {section: (result.finished, result.collisions) for section, result in probes.items()}
probe_pairs = {section: section_p99(result.telemetry, race_day_track)[section] for section, result in probes.items()}
probe_rows = {section: rows for section, (rows, _) in probe_pairs.items()}
race_day_mu_g = tuple(None if p99 is None else round(p99, 2) for _, p99 in (probe_pairs[s] for s in range(6)))
kind = {}
for section, (rows, p99) in probe_pairs.items():
    if p99 is None:
        kind[section] = "no estimate"
    elif probe_outcome[section][1] >= 1:
        kind[section] = "estimate"
    else:
        kind[section] = "lower bound"
probe_races = sum(1 for track, _ in race_log if track == RACE_DAY_TRACK)

print(f"{race_day_track}")
print("section  finished  collisions  rows  mu*g (m/s^2)  kind")
for section in range(6):
    finished, collisions = probe_outcome[section]
    print(f"  {section}      {str(finished):8s}  {collisions:10d}  {probe_rows[section]:4d}  {race_day_mu_g[section]:12.2f}"
          f"  {kind[section]}  (probe time {probes[section].total_time:.2f} s)")
lowest_training = {name: min(values) for name, values in MEASURED_MU_G.items()}
print("race-day track: lowest", min(race_day_mu_g), "in section", int(np.argmin(race_day_mu_g)),
      "; mean", round(float(np.mean(race_day_mu_g)), 2))
for name, values in MEASURED_MU_G.items():
    print(f"{name:20s} lowest {lowest_training[name]:.2f} (section {int(np.argmin(values))}), "
          f"mean {float(np.mean(values)):.2f}")

# the book's pins (plan roar-011 CPB.2): the six estimates, section 0's probe never slid, the others crashed
assert race_day_mu_g == (9.10, 9.55, 7.79, 9.35, 9.00, 7.76)
assert probe_outcome[0] == (True, 0) and kind[0] == "lower bound"
assert all(not probe_outcome[s][0] and probe_outcome[s][1] >= 1 and kind[s] == "estimate" for s in range(1, 6))
assert min(probe_rows.values()) > MIN_ROWS
assert probe_races == 6 and len(race_log) == 6

**Written answer.**
Which section's value is a lower bound, and why does the probe's outcome make it one?
The planner will use it as if it were the grip: is that a risk for race day, or a cost, and why?
Compare the race-day track's grip with the training tracks' (`MEASURED_MU_G`): where is it lowest?

**Answer.**
Section 0's value, 9.10 m/s², is a **lower bound**: its probe finished the lap (109.40 s) with 0 collisions, so with the knob at 3.5 the car never slid there.
The 99th percentile of |a| in section 0 is the grip the car *used*, not the grip it *had*, so the true μg there is at least 9.10 m/s² and may be more.
Sections 1–5 are **estimates**: each probe crashed (12, 7, 5, 3 and 3 collisions) and did not finish in 300 s, so each p99 comes from a car that slid in that section; every section had 265 to 1250 qualifying rows, far above the 20 needed.
Planning with a lower bound is a **cost**, not a risk: the planner asks section 0 for a share of 9.10 m/s², which is no more than the true grip, so the car can only be slower there than it could be, never asked for grip it does not have.
Against the training tracks the race-day grip is middling on average (mean 8.76 m/s², against 8.43–9.08) but low in two sections: section 5 (7.76, the lowest) and section 2 (7.79).
Both are below every training track's lowest section except `test-circuit-seed1`'s 7.56 (its section 5), while `-seed2` to `-seed4` never go below 8.15.

## Question 2 - Solve its line and plan the lap

Write Unit 9's line solver, `solve_line(track, margin=MARGIN, pin=PIN)`, returning the control values as a NumPy array:

1. the basis `spline_basis(n)` (n × m, m = n // 10), the left normals `normals(track.x, track.y)`, and the smoothness system: A is the rows of `second_difference(nx[:, None] * basis)` stacked **above** the rows of `second_difference(ny[:, None] * basis)`, and b = −(the n entries of `second_difference(track.x)` followed by the n entries of `second_difference(track.y)`);
2. the bounds: control value k's support is the waypoints where column k of the basis is > 0; its bound h_k is the smallest `track.half_width[i] - margin` over its support, except that h_k = 0 when its support contains any of the `pin` waypoints at each end of the lap (indices 0 … pin − 1 and n − pin … n − 1);
3. `Q, R = np.linalg.qr(A)`, then `lsq_linear(R, Q.T @ b, bounds=(-h - 1e-12, h + 1e-12), method="bvls")`, and return its `.x`.

Then write Unit 7's planner for a line, `plan_lap(track, offsets, mu_g, fraction)`, returning the planned time of one lap (s), the way the agents plan:

1. the line's points: x'_i = x_i + `offsets[i]`·nx_i and y'_i = y_i + `offsets[i]`·ny_i, with (nx, ny) the track's left normals;
2. on the line's points: κ_i = `menger(x', y')`, Δs_i the length of the segment from point i to point i + 1 (the last segment closes the lap), and the line's grip sections: point i is in section min(⌊6·d_i / L⌋, 5), with d_i the distance along the line from point 0 to point i and L the line's lap length;
3. the cap of every point, min(`TOP_SPEED`, √(`fraction`·`mu_g[section]` / |κ_i|)) where |κ_i| > 1e-9, and `TOP_SPEED` elsewhere;
4. the backward pass, twice round the lap: for i from n − 1 **down** to 0, v_i = min(v_i, √(v_{(i+1) mod n}² + 2·`A_DEC`·Δs_i));
5. then the forward pass, twice round the lap: for i from 0 **up** to n − 1, v_{(i+1) mod n} = min(v_{(i+1) mod n}, √(v_i² + 2·`A_ACC`·Δs_i));
6. the planned lap time, the sum over the n segments of Δs_i / ((v_i + v_{(i+1) mod n}) / 2).

Plan both tracks at Unit 11's shipped fraction, `START_FRACTION` (0.54): `test-circuit` with your submission's shipped line and grip (`SHIPPED_LINE`, `SHIPPED_MU_G`), and the race-day track with your line and your `race_day_mu_g`.
(Question 5 plans the race-day track again, at the fraction you freeze.)

Define:

- `solve_line`, `plan_lap`: the functions;
- `race_day_control`: your race-day line's control values, as a list of floats (it becomes the frozen file's `LINE_CONTROL`);
- `race_day_offsets`: `line_offsets(race_day_control, len(race_day_track))`;
- `inside_margin`: `True` when every |offset| is at most `race_day_track.half_width - MARGIN` (to within 1e-9);
- `planned_lap_054`: a dict from `"test-circuit"` and `RACE_DAY_TRACK` to the planned lap time at 0.54 (s);
- `fig`, `ax`: the race-day line's offsets against `race_day_track.s` (the distance along the centreline), with ± (half-width − `MARGIN`) drawn for comparison, axis labels and a title.

In [ ]:
def solve_line(track, margin=MARGIN, pin=PIN):
    # Unit 9's smoothest line: bounded least squares on the spline's control values (QR first).
    n = len(track)
    basis = spline_basis(n)
    m = basis.shape[1]
    nx, ny = normals(track.x, track.y)
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(track.x), second_difference(track.y)])
    pinned = [i for i in range(n) if i < pin or i >= n - pin]
    h = np.zeros(m)
    for k in range(m):
        support = np.nonzero(basis[:, k] > 0)[0]
        if True in [i in support for i in pinned]:
            h[k] = 0.0
        else:
            h[k] = min(track.half_width[i] - margin for i in support)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-h - 1e-12, h + 1e-12), method="bvls").x


def line_points(track, offsets):
    nx, ny = normals(track.x, track.y)
    return track.x + offsets * nx, track.y + offsets * ny


def lap_speeds(track, offsets, mu_g, fraction):
    # Unit 7's profile on the line: (v, cap, ds) at every point, as the agents plan.
    x, y = line_points(track, offsets)
    n = len(x)
    kappa = menger(x, y)
    ds = np.hypot(np.roll(x, -1) - x, np.roll(y, -1) - y)  # segment i: point i to point i + 1
    d = np.concatenate([[0.0], np.cumsum(ds)[:-1]])  # the distance along the line to point i
    lap = float(np.sum(ds))
    section = [min(math.floor(6 * d[i] / lap), 5) for i in range(n)]
    cap = [min(TOP_SPEED, math.sqrt(fraction * mu_g[section[i]] / abs(kappa[i]))) if abs(kappa[i]) > 1e-9
           else TOP_SPEED for i in range(n)]
    v = list(cap)
    for _ in range(2):  # the backward pass, twice round the lap
        for i in range(n - 1, -1, -1):
            v[i] = min(v[i], math.sqrt(v[(i + 1) % n] ** 2 + 2.0 * A_DEC * ds[i]))
    for _ in range(2):  # the forward pass, twice round the lap
        for i in range(n):
            v[(i + 1) % n] = min(v[(i + 1) % n], math.sqrt(v[i] ** 2 + 2.0 * A_ACC * ds[i]))
    return np.array(v), np.array(cap), ds


def plan_lap(track, offsets, mu_g, fraction):
    v, _, ds = lap_speeds(track, offsets, mu_g, fraction)
    return float(np.sum(ds / ((v + np.roll(v, -1)) / 2.0)))


race_day_control = [float(value) for value in solve_line(race_day_track)]
race_day_offsets = line_offsets(race_day_control, len(race_day_track))
inside_margin = float(np.max(np.abs(race_day_offsets) - (race_day_track.half_width - MARGIN))) <= 1e-9
test_circuit = load_track("test-circuit")
test_circuit_offsets = line_offsets(SHIPPED_LINE, len(test_circuit))
planned_lap_054 = {
    "test-circuit": plan_lap(test_circuit, test_circuit_offsets, SHIPPED_MU_G, START_FRACTION),
    RACE_DAY_TRACK: plan_lap(race_day_track, race_day_offsets, race_day_mu_g, START_FRACTION),
}

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(race_day_track.s, race_day_track.half_width - MARGIN, color="grey", linestyle="--", linewidth=1,
        label="+/- (half-width - 2.5 m)")
ax.plot(race_day_track.s, -(race_day_track.half_width - MARGIN), color="grey", linestyle="--", linewidth=1)
ax.plot(race_day_track.s, race_day_offsets, label="the race-day line's offset")
ax.set_xlabel("distance along the centreline, s (m)")
ax.set_ylabel("offset from the centreline (m, left positive)")
ax.set_title(f"{RACE_DAY_TRACK}: the smoothest line inside the margin")
ax.legend()
plt.show()

# the line and both plans, and why the plans differ
line_x, line_y = line_points(race_day_track, race_day_offsets)
line_length = float(np.sum(np.hypot(np.roll(line_x, -1) - line_x, np.roll(line_y, -1) - line_y)))
line_tightest = 1.0 / float(np.max(np.abs(menger(line_x, line_y))))
tc_x, tc_y = line_points(test_circuit, test_circuit_offsets)
tc_length = float(np.sum(np.hypot(np.roll(tc_x, -1) - tc_x, np.roll(tc_y, -1) - tc_y)))
tc_tightest = 1.0 / float(np.max(np.abs(menger(tc_x, tc_y))))
print(f"{len(race_day_control)} control values; inside the margin: {inside_margin}; "
      f"max |offset| {float(np.max(np.abs(race_day_offsets))):.2f} m")
print(f"race-day line: lap {line_length:.1f} m (centreline {race_day_track.length:.1f} m), "
      f"tightest radius {line_tightest:.1f} m (centreline {1.0 / float(np.max(np.abs(menger(race_day_track.x, race_day_track.y)))):.1f} m)")
print(f"test-circuit line: lap {tc_length:.1f} m, tightest radius {tc_tightest:.1f} m")
for name, lap in planned_lap_054.items():
    print(f"planned lap at 0.54 on {name}: {lap:.2f} s (three laps {3 * lap:.2f} s)")
gap_054 = planned_lap_054[RACE_DAY_TRACK] - planned_lap_054["test-circuit"]
print(f"race-day minus test-circuit: {gap_054:+.2f} s a lap")
# what each change alone does: the race-day plan at test-circuit's grip (section by section)
plan_tc_grip = plan_lap(race_day_track, race_day_offsets, SHIPPED_MU_G, START_FRACTION)
print(f"race-day line at test-circuit's grip: {plan_tc_grip:.2f} s a lap")

# independent checks: the solver's bounds hold, the profile keeps both passes, and the reference ships this line
v, cap, ds = lap_speeds(race_day_track, race_day_offsets, race_day_mu_g, START_FRACTION)
assert len(race_day_control) == len(race_day_track) // 10 == 131 and inside_margin
assert float(np.max(np.abs(race_day_offsets[:PIN]))) < 1e-9 and float(np.max(np.abs(race_day_offsets[-PIN:]))) < 1e-9
assert float(np.max(v - cap)) <= 1e-9
assert float(np.max(v ** 2 - np.roll(v, -1) ** 2 - 2.0 * A_DEC * ds)) <= 1e-9
assert float(np.max(np.roll(v, -1) ** 2 - v ** 2 - 2.0 * A_ACC * ds)) <= 1e-9
assert len(race_day_track) == 1316 and abs(race_day_track.length - 2631.2) <= 0.1
reference = submission_module(AGENTS / "checkpoint-b.py")  # the book's frozen reference: the same line
assert float(np.max(np.abs(np.array(race_day_control) - np.array(reference.LINE_CONTROL)))) < 1e-9
assert tuple(reference.MU_G) == race_day_mu_g

**Written answer.**
Is the race-day track faster or slower on paper than `test-circuit` at the same fraction, by how much, and why (its length, its corners, its grip)?
A plan says how fast the car is asked to go, not whether it can: in one or two sentences, what can the plan not tell you about racing 0.54 on this track with a car you have never met?

**Answer.**
On paper the race-day track is **faster**: at 0.54 its planned lap is 93.11 s against `test-circuit`'s 94.67 s, 1.56 s a lap (279.34 s against 284.02 s for three laps).
Two things make it faster: its line is shorter (2607.4 m against 2636.2 m on `test-circuit`'s line, 28.8 m less), and its grip is higher in most sections (mean 8.76 against 8.43 m/s²).
Planned on the same race-day line with `test-circuit`'s grip it takes 94.06 s, so the grip is worth about 0.95 s of the gap and the line about 0.61 s: shorter, and with a slightly looser tightest corner (radius 16.3 m against 15.2 m on `test-circuit`'s line), though the two low-grip sections (2 and 5) are planned slower than `test-circuit` would plan them.
The plan cannot tell whether a car can drive it: it uses the *measured* grip of the nominal car, so it says nothing about how a sealed car with less grip, a weaker throttle or a different steering will track the line at 0.54, or how often it will crash (and the fallback's cost when it does).

## Question 3 - Two rules, one record

Unit 11 gave you two ways to choose a fraction from a family's record:

- **the expected race time** (Unit 11 §2): the mean race time over all the family's races, each race counted at its time (a race not finished at the cap), and pick the fraction with the shortest;
- **the clean rule and the margin rule** (Units 7 and 11): the **edge** is the highest fraction clean on every race of the family, and the margin rule ships one grid step (`FRACTION_STEP`) inside it.

The cell below is the **reference agent's** record on the track × car family: the probe record of plan `roar-011`, `agents/unit-11.py` on the 24 training cars × the five training tracks (120 races per fraction), each track with its own line and measured grip, race seed k mod 5 for car k, the 700 s cap.
Per fraction it gives the number of clean races and their mean time, and every race that was not clean, as (track, car, race time).
Every not-clean race crashed once, fell back, and finished, under the cap.

Before racing anything, apply both rules to this record.

Write:

- `expected_time(n_clean, clean_mean, not_clean_times)`: the mean over all the races of a fraction, the clean ones at `clean_mean` (their mean is all you need) and each not-clean one at its time.

Define:

- `expected_time`: the function;
- `expected`: a dict from each fraction to its expected race time (s);
- `expected_pick`: the fraction with the shortest expected race time;
- `not_clean_rate`: a dict from each fraction to the share of its races that were not clean;
- `family_edge`: the highest fraction that is clean on all 120 races;
- `margin_pick`: the margin rule's fraction, `round(family_edge - FRACTION_STEP, 2)`.

In [ ]:
FAMILY_RACES = 120  # 24 training cars x 5 training tracks, per fraction
# The probe record of plan roar-011: agents/unit-11.py on the track x car family (race seed k mod 5, 700 s cap).
CLEAN_RECORD = {  # fraction: (clean races, their mean race time in s)
    0.56: (110, 274.50),
    0.54: (118, 276.94),
    0.52: (118, 279.52),
    0.50: (120, 282.21),
    0.48: (120, 285.11),
}
NOT_CLEAN_RECORD = {  # fraction: every race that was not clean, as (track, car, race time in s); all finished
    0.56: [("test-circuit", 8, 484.45), ("test-circuit", 9, 391.90),
           ("test-circuit-seed2", 1, 498.60), ("test-circuit-seed2", 8, 407.25), ("test-circuit-seed2", 9, 319.00),
           ("test-circuit-seed2", 10, 496.30), ("test-circuit-seed2", 13, 499.30),
           ("test-circuit-seed2", 16, 316.95), ("test-circuit-seed2", 17, 500.10),
           ("test-circuit-seed2", 20, 324.75)],
    0.54: [("test-circuit-seed2", 13, 501.00), ("test-circuit-seed2", 16, 413.40)],
    0.52: [("test-circuit-seed2", 9, 508.35), ("test-circuit-seed2", 16, 416.60)],
    0.50: [],
    0.48: [],
}

In [ ]:
def expected_time(n_clean, clean_mean, not_clean_times):
    # Unit 11 §2: every race counted at its time, the clean ones at their mean.
    return (n_clean * clean_mean + sum(not_clean_times)) / (n_clean + len(not_clean_times))


expected = {fraction: expected_time(*CLEAN_RECORD[fraction], [time for _, _, time in NOT_CLEAN_RECORD[fraction]])
            for fraction in CLEAN_RECORD}
expected_pick = min(expected, key=expected.get)
not_clean_rate = {fraction: len(NOT_CLEAN_RECORD[fraction]) / FAMILY_RACES for fraction in CLEAN_RECORD}
family_edge = max(fraction for fraction, (n_clean, _) in CLEAN_RECORD.items() if n_clean == FAMILY_RACES)
margin_pick = round(family_edge - FRACTION_STEP, 2)

print("fraction  clean  not clean  expected (s)  clean mean (s)")
for fraction in CLEAN_RECORD:
    print(f"  {fraction:.2f}    {CLEAN_RECORD[fraction][0]:3d}   {not_clean_rate[fraction]:8.1%}  {expected[fraction]:10.2f}"
          f"  {CLEAN_RECORD[fraction][1]:12.2f}")
print(f"expected time picks {expected_pick}; the family edge is {family_edge}, the margin rule picks {margin_pick}")
crash_cost_054 = float(np.mean([time for _, _, time in NOT_CLEAN_RECORD[0.54]])) - CLEAN_RECORD[0.54][1]
step_gain = CLEAN_RECORD[0.48][1] - CLEAN_RECORD[0.54][1]
break_even = step_gain / crash_cost_054
print(f"0.48 -> 0.54 saves {step_gain:.2f} s on a clean race; a crash at 0.54 costs {crash_cost_054:.0f} s on average; "
      f"0.54 beats 0.48 only while fewer than {break_even:.1%} of races crash")

# the book's pins (plan roar-011 p3, from the unrounded races: the clean means given here are rounded to 0.01 s,
# so 0.54 computes to 279.94 s against the probe's 279.95 s), and every count adds up to 120
pinned = {0.56: 286.95, 0.54: 279.95, 0.52: 282.57, 0.50: 282.21, 0.48: 285.11}
for fraction, value in pinned.items():
    assert abs(expected[fraction] - value) <= 0.01 + 1e-9
    assert CLEAN_RECORD[fraction][0] + len(NOT_CLEAN_RECORD[fraction]) == FAMILY_RACES
assert expected_pick == 0.54 and family_edge == 0.50 and margin_pick == 0.48
assert len(race_log) == 6  # nothing raced in this question

**Written answer.**
The two rules pick different fractions from the same record.
Given the bar (all five race-day races clean, median at most 300 s), which rule fits race day, and why?
In two or three sentences: what does the expected-time rule assume about how often a car crashes on race day, and what do you know about the race-day track and cars that this record does not show?
Answer before you race anything: this is your choice, and Question 6 asks you to look back at it.

**Answer.**
The expected race times are 286.95 s (0.56), 279.94 s (0.54), 282.57 s (0.52), 282.21 s (0.50) and 285.11 s (0.48), so expected time picks **0.54** (279.94 s from the rounded table; the probe record's unrounded races give 279.95 s).
The clean rule finds the family edge at **0.50** (the highest fraction clean on all 120 races), and the margin rule picks **0.48**.
The bar is all five races clean, and a crash fails it even when the fallback brings the car home (the two crashes at 0.54 took 501.00 and 413.40 s), so the clean + margin rule fits race day: I choose **0.48**, and it costs about 8.17 s a clean race against 0.54 (285.11 against 276.94 s).
The expected-time rule assumes the race-day crash rate is the family's, 2 races in 120 (1.7%) at 0.54: at that rate five races are all clean 92% of the time, and 0.54 beats 0.48 on time while fewer than 4.5% of races crash.
But the race-day track is not in the family (a new shape, a new line, two sections at 7.76 and 7.79 m/s², lower than all but one training track's lowest, and section 0 only a lower bound), and the sealed cars are new draws: the record cannot show their crash rate, and its crashes already sit on one track (`-seed2`), so the rate changes from track to track.

## Question 4 - Tune your agent on the family

The record of Question 3 is the reference's; your agent's edge may be somewhere else.
Find it with the **small protocol** (Unit 11's Exercise 8, on all five training tracks), on **your** submission:

- **the cars:** the three lowest-grip training cars, from `perturbation_factors(k)["grip"]` over `TRAINING_CARS`, lowest first;
- **the order:** the five training tracks in the order of `TRAINING`, and on each track the three cars in that order: 15 races per fraction;
- **each race:** a timing-off copy of your submission at the fraction, raced with `race(source, track, k, seed=PROTOCOL_SEED)`.
  On `test-circuit` only `MU_FRACTION` changes (the shipped line and grip); on every other training track write in that track's line, `solve_line` of it, and its grip, `MEASURED_MU_G[track]`.
  A race is judged by the clean rule against the line it raced;
- **stop each fraction at its first race that is not clean:** the fraction is then not clean on the family, and the races after it would only cost time.

Start at `START_FRACTION` (0.54). **Every outcome has a defined path:**

- **0.54 is not clean:** go down by `FRACTION_STEP` until a fraction is clean on all 15 races: that is the **edge**.
  Ship one step below it.
- **0.54 is clean on all 15:** go up by `FRACTION_STEP` until a fraction is not clean; the edge is the highest clean one; ship one step below it.
  If 0.60 (`CEILING`) is clean on all 15, stop there: the edge is 0.60, and you ship 0.58.
- **Nothing is clean down to 0.40 (`FLOOR`):** ship 0.40 and say so in your written answer.
  Race day is still raced, and the write-up carries the grade.

Race **one fraction per cell**, below.
Each fraction has two cells: the first nine races of `protocol_order` (`test-circuit`, `test-circuit-seed1`, `test-circuit-seed2`), then, only if those nine were clean, the last six (`test-circuit-seed3`, `test-circuit-seed4`).
Three fractions are laid out; if your path needs more, add two cells per fraction in the same pattern.

The book's stand-in, `agents/unit-11.py`, finds the edge at 0.50 and ships 0.48; your agent may not, and that is not an error.

Define, in the first cell:

- `protocol_cars`: the three cars, lowest grip first;
- `protocol_order`: the list of the 15 `(track, car)` pairs in protocol order;
- `training_lines`: a dict from each training track to the line it is raced with (`SHIPPED_LINE` on `test-circuit`, `solve_line` of the track elsewhere, each a list of floats);
- `protocol_source(fraction, track)`: a function returning the path of the timing-off copy of your submission for that fraction and track (`variant(..., MU_FRACTION=fraction, **NO_TIMING)`, plus `LINE_CONTROL` and `MU_G` off `test-circuit`);
- `protocol_races(fraction, pairs)`: a function that races the `(track, car)` pairs in order and stops at the first race that is not clean; it records every race it runs in `protocol_record` and returns `(all_clean, results)`, with `results` the list of the races it ran;
- `protocol_record`: a dict from `(fraction, track, car)` (the fraction rounded to 2 decimals) to `(clean, race time in s, collisions, finished)`, for every race of this question.

In [ ]:
protocol_cars = tuple(sorted(TRAINING_CARS, key=lambda k: perturbation_factors(k)["grip"])[:3])
protocol_order = [(track, k) for track in TRAINING for k in protocol_cars]
training_tracks = {name: load_track(name) for name in TRAINING}
training_lines = {name: SHIPPED_LINE if name == "test-circuit" else [float(value) for value in solve_line(track)]
                  for name, track in training_tracks.items()}
training_offsets = {name: line_offsets(training_lines[name], len(track)) for name, track in training_tracks.items()}


def protocol_source(fraction, track):
    fraction = round(fraction, 2)
    if track == "test-circuit":
        return variant(SUBMISSION, f"protocol-{fraction}-{track}", MU_FRACTION=fraction, **NO_TIMING)
    return variant(SUBMISSION, f"protocol-{fraction}-{track}", MU_FRACTION=fraction,
                   LINE_CONTROL=training_lines[track], MU_G=MEASURED_MU_G[track], **NO_TIMING)


def protocol_races(fraction, pairs):
    fraction = round(fraction, 2)
    results = []
    for track, k in pairs:
        result = race(protocol_source(fraction, track), track, k, seed=PROTOCOL_SEED)
        is_clean = clean(result, training_offsets[track])
        protocol_record[(fraction, track, k)] = (is_clean, result.total_time, result.collisions, result.finished)
        results.append(result)
        if not is_clean:
            return False, results  # the first race that is not clean decides the fraction
    return True, results


protocol_record = {}  # every race of this question, filled by protocol_races
nine, six = protocol_order[:9], protocol_order[9:]
first_nine_clean = {}  # fraction -> were its first nine races clean


def report(fraction, pairs):
    # print the races of one fraction run on these (track, car) pairs, from protocol_record, in protocol order
    for (f, track, k), (is_clean, time, collisions, finished) in protocol_record.items():
        if f == round(fraction, 2) and (track, k) in pairs:
            print(f"  {f:.2f} {track:20s} car {k:2d}: {time:7.2f} s, finished {finished}, {collisions} collisions, "
                  f"clean {is_clean}")


print("the three lowest-grip training cars:", protocol_cars,
      [round(perturbation_factors(k)["grip"], 3) for k in protocol_cars])
for name in TRAINING:
    print(f"{name:20s} {len(training_lines[name])} control values")
assert protocol_cars == (16, 18, 8) and len(protocol_order) == 15

In [ ]:
fraction = START_FRACTION  # the first fraction: 0.54
first_nine_clean[fraction], results = protocol_races(fraction, nine)
report(fraction, nine)
last = results[-1]
print(f"{fraction}: first nine clean {first_nine_clean[fraction]}, {len(results)} races")

# the book's pin (plan roar-011 CPB.3): the 7th race, test-circuit-seed2 with car 16, is the first not clean
assert not first_nine_clean[fraction] and len(results) == 7
assert list(protocol_record)[-1] == (0.54, "test-circuit-seed2", 16)

In [ ]:
if first_nine_clean[0.54]:
    all_clean, results = protocol_races(0.54, six)
    report(0.54, six)
else:
    print("0.54 was not clean in its first nine races: the last six are not raced, and the path goes down")
next_fraction = round(0.54 - FRACTION_STEP, 2) if not first_nine_clean[0.54] else round(0.54 + FRACTION_STEP, 2)
print("next fraction:", next_fraction)
assert next_fraction == 0.52

In [ ]:
fraction = next_fraction  # the second fraction: one step down, 0.52
first_nine_clean[fraction], results = protocol_races(fraction, nine)
report(fraction, nine)
print(f"{fraction}: first nine clean {first_nine_clean[fraction]}, {len(results)} races")

# the book's pin: at 0.52 the 7th race (test-circuit-seed2, car 16) is again the first not clean
assert not first_nine_clean[fraction] and len(results) == 7
assert list(protocol_record)[-1] == (0.52, "test-circuit-seed2", 16)

In [ ]:
if first_nine_clean[0.52]:
    all_clean, results = protocol_races(0.52, six)
    report(0.52, six)
else:
    print("0.52 was not clean in its first nine races: the last six are not raced, and the path goes down")
next_fraction = round(0.52 - FRACTION_STEP, 2)
print("next fraction:", next_fraction)
assert next_fraction == 0.50

In [ ]:
fraction = next_fraction  # the third fraction: 0.50
first_nine_clean[fraction], results = protocol_races(fraction, nine)
report(fraction, nine)
print(f"{fraction}: first nine clean {first_nine_clean[fraction]}, {len(results)} races")
assert first_nine_clean[fraction] and len(results) == 9

In [ ]:
last_six_clean = None
if first_nine_clean[0.50]:
    last_six_clean, results = protocol_races(0.50, six)
    report(0.50, six)
print(f"0.50: last six clean {last_six_clean}")

# the book's pin: 0.50 is clean on all 15 protocol races, so the path stops here (down from 0.54)
assert last_six_clean and len(results) == 6
assert len([key for key in protocol_record if key[0] == 0.50]) == 15

Now freeze your choice.
Your frozen file is your submission with the three race-day constants written in, and nothing else: `MU_FRACTION`, `LINE_CONTROL` (your `race_day_control`) and `MU_G` (your `race_day_mu_g`); it keeps `MPC_BUDGET_MS` as your file ships it.
Every race-day race uses its timing-off copy.

Define:

- `fraction_clean`: a dict from each fraction you raced to `True` when it was clean on all 15 races;
- `path`: `"down"`, `"up"` or `"floor"`, the path you followed;
- `edge`: the edge (`None` on the `"floor"` path);
- `FROZEN_FRACTION`: the fraction you ship, rounded to 2 decimals;
- `FROZEN_SUBMISSION`: `variant(SUBMISSION, "frozen", MU_FRACTION=FROZEN_FRACTION, LINE_CONTROL=race_day_control, MU_G=race_day_mu_g)`, the file you submit;
- `FROZEN_RACE_COPY`: `variant(FROZEN_SUBMISSION, "frozen-timing-off", **NO_TIMING)`, the file race day races;
- `protocol_races_run`: the number of entries in `protocol_record`.

After this cell your choice is frozen: `MU_FRACTION` never changes again, and nothing in the frozen file may change once a sealed car has raced.

In [ ]:
outcomes = {fraction: [is_clean for (f, _, _), (is_clean, _, _, _) in protocol_record.items() if f == fraction]
            for fraction, _, _ in protocol_record}
fraction_clean = {fraction: len(clean_list) == len(protocol_order) and False not in clean_list
                  for fraction, clean_list in outcomes.items()}
clean_fractions = [fraction for fraction, is_clean in fraction_clean.items() if is_clean]
if not clean_fractions:
    path, edge, FROZEN_FRACTION = "floor", None, FLOOR
else:
    path = "up" if fraction_clean[START_FRACTION] else "down"
    edge = max(clean_fractions)
    FROZEN_FRACTION = round(edge - FRACTION_STEP, 2)
FROZEN_SUBMISSION = variant(SUBMISSION, "frozen", MU_FRACTION=FROZEN_FRACTION, LINE_CONTROL=race_day_control,
                            MU_G=race_day_mu_g)
FROZEN_RACE_COPY = variant(FROZEN_SUBMISSION, "frozen-timing-off", **NO_TIMING)
protocol_races_run = len(protocol_record)

deciding = [(track, k, time, collisions, finished)
            for (fraction, track, k), (is_clean, time, collisions, finished) in protocol_record.items()
            if edge is not None and fraction == round(edge + FRACTION_STEP, 2) and not is_clean]
print(f"fraction clean on all 15: {fraction_clean}")
print(f"path {path}; edge {edge}; frozen {FROZEN_FRACTION}; {protocol_races_run} protocol races")
print("the race that decided the edge (one step above it):", deciding)
frozen = submission_module(FROZEN_SUBMISSION)
print(f"frozen file: MU_FRACTION {frozen.MU_FRACTION}, MU_G {frozen.MU_G}, {len(frozen.LINE_CONTROL)} line values, "
      f"MPC_BUDGET_MS {frozen.MPC_BUDGET_MS}")

# the book's pins (plan roar-011 CPB.3): down from 0.54, the edge 0.50, ship 0.48; 7 + 7 + 15 races
assert fraction_clean == {0.54: False, 0.52: False, 0.50: True}
assert path == "down" and edge == 0.50 and FROZEN_FRACTION == 0.48
assert protocol_races_run == 7 + 7 + 15 and len(race_log) == 6 + protocol_races_run
assert frozen.MPC_BUDGET_MS == 5.0 and frozen.MU_FRACTION == reference.MU_FRACTION == FROZEN_FRACTION
assert RACE_DAY_TRACK not in [track for track, _ in race_log[6:]]  # the race-day track: only the six probes so far

**Written answer.**
Report your path, your edge and the fraction you froze, with the race that decided the edge (track, car, what happened).
Compare them with Question 3: does your small protocol find the same edge as the reference's 120-race record, and the same pick as the margin rule there?
In two or three sentences: 15 races on the three lowest-grip cars, one seed each: what could this protocol miss, and what does the margin protect you against that the family cannot show you?

**Answer.**
My path is **down**: 0.54 was not clean, nor was 0.52, and 0.50 was clean on all 15, so the edge is **0.50** and I froze **0.48**, after 7 + 7 + 15 = 29 races.
At both 0.54 and 0.52 the first six races (`test-circuit` and `-seed1`) were clean and the seventh decided it: `test-circuit-seed2` with car 16, the lowest-grip car, crashed once, fell back and finished in 415.85 s (0.54) and 414.60 s (0.52), against about 280 s for a clean race.
At 0.50 all 15 races were clean, 280.90–286.60 s.
This is the same edge as the reference's 120-race record (0.50, the highest fraction clean on all 120) and the same pick as the margin rule there (0.48); the record's 0.52 shows two crashes on `-seed2` (cars 9 and 16), the protocol only the one it stopped at.
The protocol races only the three lowest-grip cars on one seed each, so it misses cars that crash for another reason (cars 9 and 13 crash in the record and are not among them: a car is its whole set of factors, not its grip) and the noise of other seeds.
The margin protects against what no training race can show: a track whose edge sits below the family's, and cars from outside the training set, which is exactly race day.

## Question 5 - Race day

Your choice is frozen.
**First plan, then check, then race.**

Plan the race-day track at your frozen fraction with Question 2's `plan_lap`.
Then run `check_submission` on the frozen file, on the race-day track: it races racekit's default kinematic car (not a carla-like car), on the file as shipped, with the timing check (`step()`'s p99 within its budget).
Run it when nothing else is running on your machine.
It must pass before any sealed car races.
`check_submission` races only racekit's kinematic car, never a sealed car, so it tells you nothing about race day, and you may run it again.
If it fails, read its report.
A timing failure: run it again with nothing else running.
Any other failure: in a new cell just above this one, correct what it names in `race_day_control` or `race_day_mu_g` (never `FROZEN_FRACTION`: only the three constants may differ from your Unit 11 submission), then, in the same cell, recompute every Question 2 value that depends on them: `race_day_offsets = line_offsets(race_day_control, len(race_day_track))`, `inside_margin`, and `planned_lap_054[RACE_DAY_TRACK]` at `START_FRACTION` (Question 5 plans, compares and judges every race with them).
Then run the freeze cell of Question 4 again, which rewrites both `FROZEN_SUBMISSION` and `FROZEN_RACE_COPY`, run this cell again (its plan and check use the corrected values; none of these cells races a carla-like car), and say what you changed in this question's written answer.
If it still does not pass, ask your teacher before any sealed car races.

Define:

- `planned_lap_frozen`: `plan_lap(race_day_track, race_day_offsets, race_day_mu_g, FROZEN_FRACTION)` (s);
- `planned_race_frozen`: the planned time of a three-lap race, `3 * planned_lap_frozen` (s);
- `submission_report`: `check_submission(FROZEN_SUBMISSION, track=RACE_DAY_TRACK)`, with `submission_report.ok` `True`.

In [ ]:
planned_lap_frozen = plan_lap(race_day_track, race_day_offsets, race_day_mu_g, FROZEN_FRACTION)
planned_race_frozen = 3 * planned_lap_frozen
submission_report = check_submission(FROZEN_SUBMISSION, track=RACE_DAY_TRACK)
print(submission_report)
print(f"planned lap at {FROZEN_FRACTION}: {planned_lap_frozen:.2f} s; three laps {planned_race_frozen:.2f} s "
      f"(at 0.54: {3 * planned_lap_054[RACE_DAY_TRACK]:.2f} s, so the margin costs "
      f"{planned_race_frozen - 3 * planned_lap_054[RACE_DAY_TRACK]:.2f} s a race on paper)")

# the frozen file passes the check before any sealed car races; a lower fraction never plans faster
assert submission_report.ok
assert planned_lap_frozen > planned_lap_054[RACE_DAY_TRACK] and planned_race_frozen < TARGET_S
assert sum(1 for track, _ in race_log if track == RACE_DAY_TRACK) == 6

Now race the five sealed cars, **once**.
This is the only cell that builds a sealed car: race each car k of `SEALED_CARS` with `race(FROZEN_RACE_COPY, RACE_DAY_TRACK, k, seed=k % 5)`, in order, and do nothing else with them.
Race all five even if one is not clean: race day is the whole result.
Run this cell once.

Define:

- `race_day`: the list of the five races, in the order of `SEALED_CARS`;
- `race_day_times`: the list of their race times (s);
- `race_day_errors`: the list of their max line errors from your race-day line, `line_error(result, race_day_offsets)` (m);
- `race_day_clean`: a list of five booleans, `clean(result, race_day_offsets)` for each race;
- `race_day_median`, `race_day_mean`: the median and the mean of the five race times (s);
- `meets_bar`: `True` when all five races are clean and `race_day_median <= TARGET_S`;
- `race_minus_plan`: `race_day_median - planned_race_frozen` (s);
- `race_day_track_races`: the number of entries in `race_log` on `RACE_DAY_TRACK` (it should be exactly 11: the six probes and the five sealed races).

In [ ]:
SEALED_CARS = range(1000, 1005)
race_day = [race(FROZEN_RACE_COPY, RACE_DAY_TRACK, k, seed=k % 5) for k in SEALED_CARS]
race_day_times = [result.total_time for result in race_day]
race_day_errors = [line_error(result, race_day_offsets) for result in race_day]
race_day_clean = [clean(result, race_day_offsets) for result in race_day]
race_day_median, race_day_mean = float(np.median(race_day_times)), float(np.mean(race_day_times))
meets_bar = False not in race_day_clean and race_day_median <= TARGET_S
race_minus_plan = race_day_median - planned_race_frozen
race_day_track_races = sum(1 for track, _ in race_log if track == RACE_DAY_TRACK)

for number, result in enumerate(race_day):
    print(f"race {number + 1}: {race_day_times[number]:.2f} s, finished {result.finished}, "
          f"{result.collisions} collisions, max line error {race_day_errors[number]:.2f} m, "
          f"clean {race_day_clean[number]}")
print(f"median {race_day_median:.2f} s, mean {race_day_mean:.2f} s; meets the bar: {meets_bar}")
print(f"planned three laps {planned_race_frozen:.2f} s; race minus plan {race_minus_plan:+.2f} s; "
      f"race-day races in the log: {race_day_track_races}")

# the race against the plan: lap by lap (the first race), and the car's speed against the frozen profile
n = len(race_day_track)
frozen_profile, _, _ = lap_speeds(race_day_track, race_day_offsets, race_day_mu_g, FROZEN_FRACTION)
telemetry = race_day[0].telemetry
lap_of_row = telemetry["progress"] // n
lap_ends = [0.0] + [float(np.max(telemetry["t"][lap_of_row <= lap])) for lap in range(3)]
lap_times = [lap_ends[lap + 1] - lap_ends[lap] for lap in range(3)]
flying = (lap_of_row >= 1) & (telemetry["held"] == 0)
above_plan = float(np.mean(telemetry["v"][flying] - frozen_profile[(telemetry["progress"][flying] + 1) % n]))
print("race 1 lap times (s):", " ".join(f"{t:.2f}" for t in lap_times), f"against a planned lap of {planned_lap_frozen:.2f} s")
print(f"on laps 2-3 the car runs {above_plan:+.2f} m/s from the plan on average; "
      f"top speed {float(np.max(telemetry['v'])):.2f} m/s against the plan's {TOP_SPEED:.0f}")

# the book's pins (plan roar-011 CPB.4): all five clean, these times to a tick, median 283.90 s
pinned_times = (283.95, 283.90, 283.60, 283.75, 283.95)
assert len(race_day_times) == 5 and False not in race_day_clean and meets_bar
assert max(abs(time - pin) for time, pin in zip(race_day_times, pinned_times)) <= TICK
assert abs(race_day_median - 283.90) <= TICK and max(race_day_errors) <= 1.5
assert race_day_track_races == 11 and len(race_log) == 6 + protocol_races_run + 5

**Written answer.**
In three to five sentences, report race day: how many races were clean, the median and the mean, and whether you met the bar.
Compare the race with your plan: how far is the median from `planned_race_frozen`, and is that gap what a three-lap race from a standing start would add?
If a race was not clean, say what happened in it (when the car crashed, and how it finished) from its `RaceResult`.

**Answer.**
`check_submission` passed on the frozen file first (the default kinematic car finished in 282.25 s with 0 collisions, `step()`'s p99 about 1 ms against the 5 ms budget).
On race day all five sealed races were clean (0 collisions, max line error at most 1.33 m), in 283.95, 283.90, 283.60, 283.75 and 283.95 s: median **283.90 s**, mean 283.83 s, so the agent **meets the bar** with 16.1 s to spare.
The race is **faster** than the plan: three planned laps at 0.48 are 289.16 s, and the median is 5.26 s quicker, so the gap is not what a standing start would add.
The standing start does cost what it should (the first race's lap 1 takes 98.20 s against a planned 96.39 s), but laps 2 and 3 take 92.95 and 92.80 s, about 3.5 s under the plan each, because the car runs 0.86 m/s above its own profile on average: it reads the goal 0.3 s ahead, so it accelerates out of every corner early, and it overshoots the 45 m/s top speed to 46.26 m/s.
No race was unclean, so there is no crash to describe.

## Question 6 - What bought what

After Question 5 your teacher hands out the **race-day record**: the reference agents on the same five sealed cars, with the race-day line and grip, at several fractions, including Unit 10's agent (no fallback) at 0.54.
Use it, and your own race day, to answer in three parts.
No new races.

**The race-day record** (the Question 6 handout: the probe record of plan `roar-011`, the reference agents on the same five sealed cars, the race-day line and grip, race seed k mod 5, 700 s cap).
It is printed here, after race day, as the teacher hands it out; the checkpoint notebook never contains it.

In [ ]:
# The probe record of plan roar-011 (p4): race times in s, in the order of the sealed cars; None: did not finish.
RACE_DAY_RECORD = {
    ("Unit 11", 0.58): ([False, False, False, False, False], [316.75, 378.70, 381.00, 379.65, 379.45]),
    ("Unit 11", 0.56): ([False, True, False, False, False], [382.05, 273.10, 381.45, 381.80, 381.50]),
    ("Unit 11", 0.54): ([False, True, False, False, False], [390.35, 275.45, 386.65, 384.65, 477.00]),
    ("Unit 11", 0.52): ([False, True, False, False, False], [480.40, 278.10, 389.90, 390.40, 486.50]),
    ("Unit 11", 0.50): ([True, True, True, False, False], [281.10, 280.80, 280.55, 486.40, 394.95]),
    ("Unit 11", 0.48): ([True, True, True, True, True], [283.95, 283.90, 283.60, 283.75, 283.95]),
    ("Unit 10", 0.54): ([False, True, False, False, False], [379.85, 275.45, None, None, None]),
}
print("agent    fraction  clean  race times (s)                                  mean (s)  median (s)")
record_mean = {}
for (agent, fraction), (clean_list, times) in RACE_DAY_RECORD.items():
    finished = [time for time in times if time is not None]
    record_mean[(agent, fraction)] = float(np.mean(times)) if len(finished) == len(times) else None
    shown = ", ".join("DNF" if time is None else f"{time:.2f}" for time in times)
    mean_text = "-" if record_mean[(agent, fraction)] is None else f"{record_mean[(agent, fraction)]:.2f}"
    median_text = "-" if len(finished) < len(times) else f"{float(np.median(times)):.2f}"
    print(f"{agent:8s} {fraction:.2f}      {sum(clean_list)}/5    {shown:46s}  {mean_text:>8s}  {median_text:>9s}")

crash_times_054 = [time for is_clean, time in zip(*RACE_DAY_RECORD[("Unit 11", 0.54)]) if not is_clean]
clean_054 = [time for is_clean, time in zip(*RACE_DAY_RECORD[("Unit 11", 0.54)]) if is_clean]
print(f"at 0.54: {len(crash_times_054)} of 5 crashed, finishing in {min(crash_times_054):.2f}-{max(crash_times_054):.2f} s; "
      f"the one clean race {clean_054[0]:.2f} s")
print(f"race-day crash rate at 0.54: {len(crash_times_054) / 5:.0%}, against {not_clean_rate[0.54]:.1%} on the family; "
      f"expected time on the family {expected[0.54]:.2f} s, race-day mean {record_mean[('Unit 11', 0.54)]:.2f} s")

# the handout's means, and my race day is its 0.48 row to a tick
pinned_means = {0.58: 367.11, 0.56: 359.98, 0.54: 382.82, 0.52: 405.06, 0.50: 344.76, 0.48: 283.83}
for fraction, value in pinned_means.items():
    assert abs(record_mean[("Unit 11", fraction)] - value) <= 0.005 + 1e-9
assert max(abs(mine - theirs) for mine, theirs in zip(race_day_times, RACE_DAY_RECORD[("Unit 11", 0.48)][1])) <= TICK
assert [sum(RACE_DAY_RECORD[("Unit 11", f)][0]) for f in (0.58, 0.56, 0.54, 0.52, 0.50, 0.48)] == [0, 1, 1, 1, 3, 5]
assert len(race_log) == 6 + protocol_races_run + 5  # no new races

**Written answer.**

1. **The margin.** What did the margin buy on race day?
   Compare the fraction the margin rule picks with the fractions above it, race by race: clean count, race times, mean.
2. **The fallback.** What did the fallback buy at 0.54 (Unit 10's agent against Unit 11's, the same cars)?
   Why did it still not make 0.54 a good choice for this bar?
3. **The expected time.** Why did the expected race time over the training family misjudge race day?
   Name what the family's record could not contain.
   Say which of your Question 3 assumptions race day tested.

**Answer.**

1. **The margin.** At 0.48, the margin rule's pick, all five sealed cars were clean, 283.60–283.95 s, mean 283.83 s.
   One step up, at 0.50 (the family's edge), the first three cars were about 3 s faster (281.10, 280.80, 280.55 against 283.95, 283.90, 283.60), but cars four and five crashed and took 486.40 and 394.95 s: 3 of 5 clean, mean 344.76 s.
   Above that it gets worse: 0.52, 0.54 and 0.56 are clean on one car of five (means 405.06, 382.82, 359.98 s) and 0.58 on none (367.11 s).
   So the margin cost about 3 s a clean race and bought two clean races, 61 s of mean, and the bar: on race day the family's edge itself was not clean, so one step inside it was exactly what was needed.
   It makes a crash rarer, not impossible: on 19 more cars of the family (record only), 0.48 crashed one.
2. **The fallback.** At 0.54 Unit 11's agent finished all five races; Unit 10's, with no fallback, finished only two (379.85 and 275.45 s), and the other three never finished, crashing 5–6 times each.
   The fallback turned three did-not-finish races into finishes (384.65–477.00 s).
   It still did not make 0.54 a good choice, because it makes a crash cheaper, not rarer: four of the five cars still crashed, each crash cost 110–200 s against the clean 275.45 s, the median was 386.65 s, and a crashed race is not clean, so 0.54 fails the bar on both counts.
3. **The expected time.** The expected time over the family (279.94 s at 0.54) assumed race day would crash as the family did, 2 races in 120 (1.7%); on race day 0.54 crashed 4 of 5 (80%), and its mean was 382.82 s, 103 s worse than expected and 99 s worse than 0.48.
   The family's record could not contain the crash rate of these cars on this track: the race-day track and the sealed cars are both outside it, and the crash rate belongs to the track and the car together (with no perturbation the same track at 0.54 is clean in about 275 s, and the family's own crashes sat on one track, `-seed2`).
   Race day tested both of my Question 3 assumptions: that the family's crash rate would carry to a new track and new cars (it did not, at 0.54 or even at the edge 0.50), and that one grid step inside the family's edge would carry where the family's numbers did not (it did: 0.48 was clean on all five).